# Pramaan: everything that needs a GPU (one notebook)

Runs on a **T4** (Colab free tier is enough), in the browser or with the **Colab extension for VS Code**.

| Part | What it does | Why the project needs it | Time on a T4 |
|---|---|---|---|
| **A. OCR** | docTR (`db_resnet50` + `crnn_vgg16_bn`) on the scanned pages that have no usable text layer | Scanned filings are a large share of the data; without OCR they contribute no questions and the text layer's digit errors bound accuracy. On a CPU it takes hours. | about 1 to 2 s per page |
| **B. Reranker** | fine-tunes the larger cross-encoder (`MiniLM-L12`) on the free XBRL page labels | The CPU run already produced a `MiniLM-L6` model; this is the GPU upgrade and the spec's "notebook that runs on a T4" | minutes |

Everything else in the project (API calls, retrieval, evaluation) runs on a laptop CPU.

## How data gets in and out
Build one input file on your computer: `python scripts/make_gpu_bundle.py` produces `colab_inputs/pramaan_gpu_inputs.zip` (the code, the PDFs that need OCR, the reranker training pairs). It contains **exchange documents, so keep it private**: put it in your Google Drive, or in a *private* Hugging Face dataset repo. This notebook does not use `files.upload()` (it does not work in VS Code).

Set `TRANSFER` in the next cell: `"drive"` (default), `"hf"` (if `drive.mount` does not work in your VS Code session), or `"local"` (a path on the machine running the kernel). When it finishes it writes `pramaan_gpu_outputs.zip` next to the input; bring it back with `python scripts/import_gpu_outputs.py`.

**Quick check first:** set `OCR_LIMIT = 20` and `RUN_RERANKER = False`, run all, then run again with the full settings.

**Important:** importing OCR into a database that already has evaluation results changes which pages count as results pages, so rebuilding the gym there changes the question set and breaks comparison with earlier numbers. Use this on the September season data, or on a copy.

In [ ]:
import getpass, hashlib, json, os, pathlib, shutil, subprocess, sys, time, zipfile

# ---- settings -------------------------------------------------------------------------------
TRANSFER = "drive"     # "drive" | "hf" | "local"
DRIVE_FOLDER = "MyDrive/Projects/Pramaan/gpu"   # holds pramaan_gpu_inputs.zip; outputs are written next to it
HF_REPO = ""           # TRANSFER = "hf": a PRIVATE dataset repo such as "yourname/pramaan-gpu"
LOCAL_INPUT = ""       # TRANSFER = "local": path to pramaan_gpu_inputs.zip on the kernel's machine
DEVICE = "cuda"        # "cuda" for the real run; "cpu" only to test the notebook without a GPU
RUN_OCR = True
RUN_RERANKER = True
OCR_LIMIT = None       # for example 20, for a quick check
# ----------------------------------------------------------------------------------------------

import torch
if DEVICE == "cuda":
    assert torch.cuda.is_available(), "No GPU. Choose a Colab server with a T4 GPU (Runtime > Change runtime type in the browser), then run again."
    GPU = f"{torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_memory / 1e9:.0f} GB)"
else:
    GPU = "none (DEVICE = 'cpu', a test run)"
print("device:", DEVICE, "|", GPU)
WORK = pathlib.Path(os.environ.get("PRAMAAN_GPU_WORK", "/content/pramaan_gpu"))
WORK.mkdir(parents=True, exist_ok=True)

In [ ]:
OUT_NAME = "pramaan_gpu_outputs.zip"


def _hf_token() -> str:
    return os.environ.get("HF_TOKEN") or getpass.getpass("Hugging Face token (the repo must be a PRIVATE dataset): ")


def fetch_inputs() -> pathlib.Path:
    if TRANSFER == "drive":
        from google.colab import drive

        drive.mount("/content/drive")
        src = pathlib.Path("/content/drive") / DRIVE_FOLDER / "pramaan_gpu_inputs.zip"
    elif TRANSFER == "hf":
        from huggingface_hub import hf_hub_download

        src = pathlib.Path(hf_hub_download(HF_REPO, "pramaan_gpu_inputs.zip", repo_type="dataset", token=_hf_token()))
    elif TRANSFER == "local":
        src = pathlib.Path(LOCAL_INPUT).expanduser()
    else:
        raise ValueError("TRANSFER must be 'drive', 'hf' or 'local'")
    assert src.exists(), f"not found: {src}. Put pramaan_gpu_inputs.zip there first (python scripts/make_gpu_bundle.py)."
    return src


def deliver(path: pathlib.Path, source: pathlib.Path) -> None:
    print("result file on this machine:", path)
    if TRANSFER == "drive":
        dest = source.parent / OUT_NAME
        shutil.copy2(path, dest)
        print("copied to Drive:", dest)
    elif TRANSFER == "hf":
        from huggingface_hub import HfApi

        HfApi().upload_file(path_or_fileobj=str(path), path_in_repo=OUT_NAME, repo_id=HF_REPO, repo_type="dataset", token=_hf_token())
        print("uploaded to the private dataset repo", HF_REPO)
    else:
        dest = source.parent / OUT_NAME
        shutil.copy2(path, dest)
        print("copied next to the input:", dest)


SOURCE = fetch_inputs()
IN = WORK / "inputs"
shutil.rmtree(IN, ignore_errors=True)
with zipfile.ZipFile(SOURCE) as z:
    z.extractall(IN)
info = json.loads((IN / "bundle_manifest.json").read_text())
for name, want in info["sha256"].items():  # a damaged upload fails here, not an hour into the OCR
    h = hashlib.sha256()
    with open(IN / name, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    assert h.hexdigest() == want, f"{name} is damaged; upload it again"
print(json.dumps({k: info[k] for k in ("built_from", "ocr_pages", "ocr_filings", "ocr_reasons", "reranker_files")}, indent=1))

In [ ]:
CODE = WORK / "code"
shutil.rmtree(CODE, ignore_errors=True)
CODE.mkdir()
with zipfile.ZipFile(IN / "Pramaan.zip") as z:
    z.extractall(CODE)
inner = [p for p in CODE.iterdir() if p.is_dir() and not (CODE / "pyproject.toml").exists()]
if len(inner) == 1:  # the zip has a top-level Pramaan/ folder
    for item in inner[0].iterdir():
        shutil.move(str(item), CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(CODE / "requirements-gpu-notebook.txt")], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(CODE), "--no-deps"], check=True)
os.environ["PRAMAAN_HOME"] = str(WORK / "home")
import importlib
importlib.invalidate_caches()
if str(CODE) not in sys.path:
    sys.path.insert(0, str(CODE))
import pramaan
print("pramaan", pramaan.__version__, "from", pathlib.Path(pramaan.__file__).parent)
# If pip upgraded numpy or similar and the next cell fails with an import error, restart the kernel once
# (Runtime > Restart session) and run all again: the packages are then in place from the start.

## A. OCR with docTR on the GPU

The PDFs are rendered at 200 dpi here and each page becomes one JSON of words with boxes in page units. Pages already done are skipped, so re-running the cell after an interruption continues where it stopped (files in `/content` are lost if the runtime itself is recycled).

In [ ]:
ocr_report = None
OCR_ZIP = WORK / "ocr_results.zip"
if RUN_OCR:
    from pramaan.data.ocr_run import ocr_zip

    ocr_report = ocr_zip(IN / "ocr_inputs.zip", OCR_ZIP, "doctr", DEVICE, limit=OCR_LIMIT)
    print(ocr_report)
    with zipfile.ZipFile(OCR_ZIP) as z:
        counts = sorted(len(json.loads(z.read(n))["words"]) for n in z.namelist())
    print(f"{len(counts)} pages, median {counts[len(counts) // 2]} words per page, {sum(c == 0 for c in counts)} empty pages")
    ocr_report["pages_with_zero_words"] = sum(c == 0 for c in counts)
else:
    print("OCR skipped (RUN_OCR = False)")

## B. Reranker fine-tune

Trains on the free page labels from the train split only (the export refuses dev and test pages), holds out 10% of companies and stops early on held-out nDCG@10. The output is a small model folder; use it with `PRAMAAN_RERANKER_PATH`.

In [ ]:
rr_report = None
RR_ZIP = WORK / "reranker_model.zip"
pairs, groups = IN / "reranker" / "reranker_pairs.parquet", IN / "reranker" / "reranker_groups.parquet"
if RUN_RERANKER and pairs.exists() and groups.exists():
    from pramaan.core.config import retrieval_config
    from pramaan.retrieval.train_ce import train

    cfg = retrieval_config().training
    res = train(pairs, groups, cfg["colab_base_model"], WORK / "reranker", epochs=int(cfg["epochs"]), batch_size=int(cfg["batch_size"]), lr=float(cfg["lr"]), max_length=int(cfg["max_length"]), device=DEVICE)
    rr_report = {k: res[k] for k in ("base_holdout", "best_holdout", "best_epoch", "seconds") if k in res}
    rr_report["base_model"] = cfg["colab_base_model"]
    print(json.dumps(rr_report, indent=1, default=str))
    shutil.make_archive(str(RR_ZIP.with_suffix("")), "zip", WORK / "reranker" / "model")
else:
    print("reranker skipped (RUN_RERANKER = False, or the bundle has no reranker files)")

In [ ]:
report = {"gpu": GPU, "device": DEVICE, "transfer": TRANSFER, "ocr": ocr_report, "reranker": rr_report, "bundle": info, "finished": time.strftime("%Y-%m-%d %H:%M:%S")}
OUT = WORK / OUT_NAME
with zipfile.ZipFile(OUT, "w", zipfile.ZIP_STORED) as z:
    for part in (OCR_ZIP, RR_ZIP):
        if part.exists():
            z.write(part, part.name)
    z.writestr("run_report.json", json.dumps(report, indent=1, default=str))
print(f"{OUT.name}: {OUT.stat().st_size / 1e6:.1f} MB")
deliver(OUT, SOURCE)

## Back on your computer

```bash
python scripts/import_gpu_outputs.py ~/Downloads/pramaan_gpu_outputs.zip --home <your data home>
pramaan index && pramaan gym build && pramaan exp E1      # only on the season data or a copy, see the note at the top
```

The import loads the OCR (`pages.text_source = doctr`) and unpacks the reranker to `<home>/artifacts/reranker/gpu/model`; to use it, `export PRAMAAN_RERANKER_PATH=<that folder>` and compare with `pramaan exp E2 --finetuned <that folder>`.